In [ ]:
DateTable =
ADDCOLUMNS (
    CALENDAR (
        DATE(2024,1,1),
        DATE(2025,12,31)
    ),
    "Year", YEAR([Date]),
    "Month Number", MONTH([Date]),
    "Month Name", FORMAT([Date], "MMM"),
    "Year Month", FORMAT([Date], "YYYY-MMM"),
    "Quarter", "Q" & FORMAT([Date], "Q")
)

Transaction Date =
DATE (
    YEAR('upi_transaction upi_transaction_history'[timestamp]),
    MONTH('upi_transaction upi_transaction_history'[timestamp]),
    DAY('upi_transaction upi_transaction_history'[timestamp])
)

Total Transactions = 
COUNTROWS('upi_transaction upi_transaction_history')

Total Transaction Amount =
SUM('upi_transaction upi_transaction_history'[amount])

Average Transaction Amount =
AVERAGE('upi_transaction upi_transaction_history'[amount])

Successful Transactions =
CALCULATE (
    [Total Transactions],
    'upi_transaction upi_transaction_history'[status] = "success"
)

Failed Transactions =
CALCULATE (
    [Total Transactions],
    'upi_transaction upi_transaction_history'[status] = "failed"
)

Failure Rate =
DIVIDE (
    [Failed Transactions],
    [Total Transactions],
    0
)

Fraud Transactions =
CALCULATE (
    [Total Transactions],
    'upi_transaction upi_transaction_history'[fraud_flag] = "True"
)

Fraud Rate =
DIVIDE (
    [Fraud Transactions],
    [Total Transactions],
    0
)

Fraud Amount =
CALCULATE (
    SUM('upi_transaction upi_transaction_history'[amount]),
    'upi_transaction upi_transaction_history'[fraud_flag] = "True"
)

Reversed Transactions =
CALCULATE (
    [Total Transactions],
    'upi_transaction upi_transaction_history'[reversal_flag] = "True"
)

      
Merchant Total Transactions = 
CALCULATE(
    [Total Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Fraud Transactions = 
CALCULATE(
    [Fraud Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Fraud Rate = 
DIVIDE(
    [Merchant Fraud Transactions],
    [Merchant Total Transactions],
    0
)

High Risk Merchants = 
COUNTROWS(
    FILTER(
        VALUES('upi_transaction upi_transaction_history'[merchant_id]),
        CALCULATE([Merchant Total Transactions]) >= 50
            &&
        CALCULATE([Merchant Fraud Rate]) >= 0.05
    )
)

High Failure Devices = 
COUNTROWS(
    FILTER(
        VALUES('upi_transaction upi_transaction_history'[device_type]),
        CALCULATE([Total Transactions]) >= 50
            &&
        CALCULATE([Failure Rate]) >= 0.05
    )
)

Merchant Failure Transactions = 
CALCULATE(
    [Failed Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Failure Rate = 
DIVIDE(
    [Merchant Failure Transactions],
    [Merchant Total Transactions],
    0
)

Total Fraud Alerts =
COUNTROWS('upi_transaction fraud_alert_history')

Open Alerts =
CALCULATE (
    [Total Fraud Alerts],
    'upi_transaction fraud_alert_history'[resolved] = "False"
)

Resolved Alerts =
CALCULATE (
    [Total Fraud Alerts],
    'upi_transaction fraud_alert_history'[resolved] = "True"
)

Alert Resolution Rate =
DIVIDE (
    [Resolved Alerts],
    [Total Fraud Alerts],
    0
)


           